# 23 · E1j: E1's verdict, re-decided with the slow-current rung on the ladder

**Where Notebooks 19–22 left off.** A global slow K⁺-like current is part of the best description of the atlas that we have found:
* chosen on training pairs it gains +0.025 held-out FEVE over L1 (Notebooks 19, 21);
* it survives a consistent start from the exact resting state (21);
* it survives being **fitted** with the same optimizer and budget as a re-fitted L1: +0.024 [+0.014, +0.039] (22).

Its timescale (5–20 s) is not identifiable from amplitude data: τ_n never moved during fitting, as Notebooks 06–07 predicted. E1's nested warm start ("new mechanism nearly off") barely grows it (+0.008), which explains why E1's L2 rung found almost nothing.

**But E1's verdict is not "L1 vs something".** It is a decision over the whole ladder, under registered rules. In Notebook 04 every mechanistic rung tied with the wiring-only **L0** at ≈0.30 FEVE, and the necessity bracket was L0 ≤ closed ≤ L1 (Notebook 07); the calibrated rule v4 put the closed level at L0 (Notebook 09). So the question is whether `L1s` changes any of E1's three decisions when it is placed on the ladder between L1 and L2:

| rule | E1 notebook | asks |
|---|---|---|
| sufficiency (L\*) | 04 | the lowest rung that no deeper rung beats by more than 0.05 (upper CI) |
| necessity (L_needed) | 07 | the deepest rung that beats *every* shallower rung (lower CI > 0) |
| calibrated v4 (L_closed) | 09 | the deepest rung whose gain over every shallower rung exceeds the null-calibrated δ |

**Scoring is exactly E1's:** the cross-validated held-out predictions of Notebook 04 for L0–L4 and B, both strains, bootstrap over stimulated neurons. `L1s` (Notebook 22's fitted `L1s_sel`) enters with predictions on the same folds.

**A caution, stated before the result.** `L1s` was proposed *after* seeing held-out results: an unregistered observation in Notebook 18. Every later test was cross-validated and pre-registered, but the choice of which mechanism to test was data-driven. A change of E1's verdict here is therefore strong evidence, not confirmation. Confirmation needs new data: the 5-s pulse experiment that Notebook 08 ranked first, or recordings from held-out animals.

In [ ]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

In [ ]:
import json, pickle, time, datetime, numpy as np, matplotlib.pyplot as plt, jax, jax.numpy as jnp
import closurebench
assert tuple(int(x) for x in closurebench.__version__.split(".")[:3]) >= (0, 19, 0), \
    f"Notebook 23 needs closurebench >= 0.19.0 (found {closurebench.__version__}); copy the full package and restart the kernel"
from closurebench import data as D, ladder as lad, metrics as M_
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
N_BOOT, THRESHOLD = 2000, 0.05
TAG = "" if MODE == "full" else f"_{MODE}"
E1_TAG = "_laptop" if MODE == "smoke" else TAG
LADDER0 = ("L0", "L1", "L2", "L3", "L4"); LADDER1 = ("L0", "L1", "L1s", "L2", "L3", "L4")
try:
    e1_result = json.load(open(os.path.join(RESULTS, f"E1_result{E1_TAG}.json")))
    c = json.load(open(os.path.join(RESULTS, e1_result["preregistration"])))["config"]
    ds = D.Dataset.load(os.path.join(RESULTS, "atlas_dataset.npz"))
    if c["n_neurons"] < ds.N:
        mk = ds.mask()[0]; score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
        ds = D.subset(ds, np.sort(np.argsort(-score)[:c["n_neurons"]]))
    st = lad.Structure.from_dataset(ds, tie=c["tie_classes"]); cfg = lad.SimConfig(**c["sim"])
    folds = M_.kfold_pairs(ds, c["k_folds"], c["fold_seed"])
    PRED = {k: v for k, v in np.load(os.path.join(RESULTS, f"E1_heldout_predictions{E1_TAG}.npz")).items()}
    ck1 = os.path.join(RESULTS, f"e1_checkpoints{E1_TAG}")
    E1_MODELS = [pickle.load(open(os.path.join(ck1, f"fold{f}_L1_final.pkl"), "rb"))["params"] for f in range(c["k_folds"])]
    CK22 = os.path.join(RESULTS, "e1i_ck", "laptop" if MODE == "smoke" else MODE)
    L1S = [pickle.load(open(os.path.join(CK22, f"fold{f}_L1s_sel.pkl"), "rb")) for f in range(c["k_folds"])]
    L1R = [pickle.load(open(os.path.join(CK22, f"fold{f}_L1_refit.pkl"), "rb")) for f in range(c["k_folds"])]
    d9 = os.path.join(RESULTS, f"E1e_result{E1_TAG}.json")
    DELTA = json.load(open(d9))["delta"] if os.path.exists(d9) else {}
    SOURCE = f"E1 ({E1_TAG.strip('_') or 'full'}) + Notebook 22 fits"
except (FileNotFoundError, KeyError) as err:
    if MODE != "smoke":
        raise
    print("E1 / Notebook 22 results not found (", err, ") -> CODE-PATH TEST ONLY on a synthetic worm with made-up rung predictions")
    ds, p_true, R = lad.make_synthetic(lad.random_wiring(N=30, S=12, seed=1), true_level="L1", seed=1)
    st = lad.Structure.from_dataset(ds); cfg = lad.SimConfig(); folds = M_.kfold_pairs(ds, 3, 0)
    rng = np.random.default_rng(0); R = np.asarray(R)
    PRED = {L: R + rng.normal(size=R.shape) * s * R.std() for L, s in zip(LADDER0 + ("B",), (0.6, 0.5, 0.55, 0.6, 0.6, 0.9))}
    E1_MODELS = L1S = L1R = None; DELTA = {"repeat": 0.01}; SOURCE = "synthetic (code path only)"
print(f"source: {SOURCE} | {ds.N} neurons, {ds.S} stimuli, strains {ds.strains} | E1 rungs {sorted(PRED)} | calibrated deltas {DELTA}")

### Pre-registration (before §1)

Recorded in `results/E1j_preregistration*.json`.
* The metric is E1's primary metric: pooled held-out FEVE over all observed pairs, both strains, with 2000 bootstrap draws resampling stimulated neurons.
* All three of E1's rules are applied to the original ladder (L0–L4, B) and to the ladder with `L1s` inserted between L1 and L2, using E1's thresholds: 0.05 for sufficiency, 0 for necessity, and Notebook 09's calibrated δ (and δ = 0) for v4.

The rules:
* **SAME_FOLDS** (must hold): the L1 predictions rebuilt from the fold models match Notebook 04's saved held-out predictions (max difference < 10⁻³), so `L1s` is scored on exactly E1's folds.
* **CLOSED_LEVEL_CHANGES:** at least one of the three decisions differs between the two ladders.
* Reported:
  * `L1s` − L0, `L1s` − L1 and `L1s` − B, with CIs;
  * the same decisions with L1 replaced by Notebook 22's re-fitted L1 (equal training budget);
  * the decisions on WT pairs only.

In [ ]:
prereg = os.path.join(RESULTS, f"E1j_preregistration{TAG}.json")
plan = {"experiment": "E1j E1 re-decided with L1s", "mode": MODE, "registered_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source": SOURCE, "ladders": {"original": list(LADDER0), "with_L1s": list(LADDER1)}, "n_boot": N_BOOT,
        "rules": {"sufficiency": f"closure_decision, threshold {THRESHOLD}", "necessity": "metrics.necessity",
                  "v4": f"calibrated_closure with Notebook 09 deltas {DELTA} and delta = 0"},
        "SAME_FOLDS": "rebuilt L1 held-out predictions match Notebook 04's (max |diff| < 1e-3)",
        "CLOSED_LEVEL_CHANGES": "any of the three decisions differs between the two ladders (all strains)"}
if os.path.exists(prereg):
    plan = json.load(open(prereg)); print("Existing registration (", plan["registered_at"], ") used unchanged.")
else:
    json.dump(plan, open(prereg, "w"), indent=2); print("Registered ->", prereg)

## 1 · `L1s` held-out predictions on E1's folds

In [ ]:
def cv_from(models_by_fold, level):
    out = np.zeros(ds.mean.shape[:3])
    for f, pp in enumerate(models_by_fold):
        R = np.asarray(lad.predict(level, jax.tree_util.tree_map(jnp.asarray, pp), st, ds.stim, ds.strains, cfg, np.arange(ds.S)))
        out[folds[f]] = R[folds[f]]
    return out
if L1S is not None:
    t0 = time.time()
    L1_rebuilt = cv_from(E1_MODELS, "L1")
    m = ds.mask() & np.any(np.stack(folds), 0)
    dmax = float(np.abs(L1_rebuilt[m] - PRED["L1"][m]).max())
    SAME_FOLDS = dmax < 1e-3
    print(f"SAME_FOLDS: max |rebuilt L1 - Notebook 04's L1| on held-out pairs = {dmax:.2e} -> {SAME_FOLDS}")
    PRED["L1s"] = cv_from(L1S, "L1s"); PRED["L1_refit"] = cv_from(L1R, "L1")
    print(f"L1s and re-fitted L1 predictions assembled ({time.time() - t0:.0f} s)")
else:
    SAME_FOLDS = True
    PRED["L1s"] = PRED["L1"] * 1.05; PRED["L1_refit"] = PRED["L1"]
assert SAME_FOLDS, "fold mismatch: L1s would not be comparable with E1's rungs"

## 2 · E1's three rules, with and without `L1s`

In [ ]:
def decide(ladder, strain=None, l1_key="L1"):
    preds = {L: PRED[l1_key if L == "L1" else L] for L in ladder + ("B",)}
    point, boots = M_.bootstrap_feve(preds, ds, list(ladder) + ["B"], n_boot=N_BOOT, seed=0, strain=strain)
    suff = M_.closure_decision(point, boots, ladder=ladder, threshold=THRESHOLD, blackbox="B")
    nec = M_.necessity(point, boots, ladder=ladder)
    v4 = {"delta=0": M_.calibrated_closure(boots, 0.0, ladder)["L_closed"]}
    v4.update({f"delta[{k}]={d:.3f}": M_.calibrated_closure(boots, d, ladder)["L_closed"] for k, d in DELTA.items()})
    return dict(point=point, boots=boots, L_star=suff["L_star"], L_needed=nec["L_needed"], v4=v4, nec=nec, suff=suff)
RES = {}
for name, lad_, strain, l1k in (("original", LADDER0, None, "L1"), ("with_L1s", LADDER1, None, "L1"),
                                 ("with_L1s_refitL1", LADDER1, None, "L1_refit"), ("original_wt", LADDER0, "wt", "L1"), ("with_L1s_wt", LADDER1, "wt", "L1")):
    RES[name] = decide(lad_, strain, l1k)
    r = RES[name]
    print(f"{name:17s}: FEVE " + " ".join(f"{L}={r['point'][L]:.3f}" for L in lad_ + ('B',)))
    print(f"{'':17s}  sufficiency L* = {r['L_star']} | necessity L_needed = {r['L_needed']} | v4 L_closed = {r['v4']}")
print()
print(M_.format_necessity(RES["with_L1s"]["nec"]))
b = RES["with_L1s"]["boots"]; pt = RES["with_L1s"]["point"]
for x, y in (("L1s", "L0"), ("L1s", "L1"), ("L1s", "B"), ("L1", "L0")):
    d = b[x] - b[y]; lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"  {x} - {y}: {pt[x] - pt[y]:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
keys = ("L_star", "L_needed")
CHANGES = {k: (RES["original"][k], RES["with_L1s"][k]) for k in keys}
CHANGES.update({f"v4 {k}": (RES["original"]["v4"][k], RES["with_L1s"]["v4"][k]) for k in RES["original"]["v4"]})
CLOSED_LEVEL_CHANGES = any(a != b_ for a, b_ in CHANGES.values())
print("\ndecision (original -> with L1s):", CHANGES)
print("Registered rules -> SAME_FOLDS:", SAME_FOLDS, "| CLOSED_LEVEL_CHANGES:", CLOSED_LEVEL_CHANGES)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
lab = list(LADDER1) + ["B"]
mu = [pt[L] for L in lab]; lo = [np.percentile(b[L], 2.5) for L in lab]; hi = [np.percentile(b[L], 97.5) for L in lab]
ax.errorbar(range(len(lab)), mu, yerr=[np.subtract(mu, lo), np.subtract(hi, mu)], fmt="o", capsize=3)
ax.set_xticks(range(len(lab))); ax.set_xticklabels(lab); ax.set_ylabel("held-out FEVE (all strains, 95% CI)")
ax.axhline(0, c="0.6", lw=0.6); plt.tight_layout(); plt.show()

In [ ]:
out = {"source": SOURCE, "SAME_FOLDS": SAME_FOLDS, "CLOSED_LEVEL_CHANGES": CLOSED_LEVEL_CHANGES, "changes": CHANGES,
       "decisions": {k: {"feve": {L: float(v) for L, v in r["point"].items()}, "L_star": r["L_star"], "L_needed": r["L_needed"], "v4": r["v4"]}
                     for k, r in RES.items()}}
json.dump(out, open(os.path.join(RESULTS, f"E1j_result{TAG}.json"), "w"), indent=2, default=float)
print("saved", f"E1j_result{TAG}.json")

## 3 · Reading the result

| outcome | reading for C2 |
|---|---|
| CLOSED_LEVEL_CHANGES (necessity or v4 move to `L1s`) | `L1s` beats every shallower rung, the wiring-only L0 included. The closed level, as far as single-neuron stimulation can tell, includes **slow (seconds-scale) dynamics** beyond wiring, signs and a static nonlinearity. That is the first mechanism E1 can positively require. Confirm with new data (5-s pulses, Notebook 08). |
| no change, `L1s` beats L1 but not L0 | `L1s` improves the *mechanistic* L1 but not beyond the linear wiring-only model. What it adds may be what L0 already captures in another way (L0 has its own time constant). The bracket stays L0 ≤ closed ≤ L1. |
| no change, sufficiency only | The gain is below E1's 0.05 sufficiency margin: detectable, but not large enough to move L\*. |

**Limits.**
* The choice of mechanism was data-driven (Notebook 18); see the caution above.
* `L1s` had E1's L1 budget plus 300 steps. The re-fitted-L1 row checks that this extra budget is not the reason.
* 120 neurons (laptop E1). The 300-neuron E1 has not been run.